**Imports**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

**Reading The Table**

In [0]:
fact_rental_df= spark.read.table("rental.bike_rental_silver.Rental")

In [0]:
dim_membership = spark.read.table("rental.bike_rental_silver.membership")

In [0]:
dim_membership = spark.read.table("rental.bike_rental_silver.membership").select("customer_id","id").withColumnRenamed("id","membership_id")

**Adding Membership_ID**

In [0]:
fact_rental_df = fact_rental_df.join(dim_membership, on ="customer_id")

**Adding Date ID**

In [0]:
fact_rental_df = fact_rental_df.withColumn("date_id", F.concat(F.col("rental_year"), F.lpad(F.col("rental_month"),2,"0"), F.lpad(F.col("rental_day"),2,"0")).cast("int"))

In [0]:
fact_rental_df = fact_rental_df.select("rental_id","customer_id","bike_id","duration","total_paid","membership_id","date_id")

**Rental Count**

In [0]:
window = Window.partitionBy("customer_id")
fact_rental_df = fact_rental_df.withColumn("rental_count", F.count("customer_id").over(window))

**Writing to Gold**

In [0]:
fact_rental_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("rental.bike_rental_gold.fact_rentals")